In [6]:
# ============================================================
# CELL 1 — VERIFY COLAB T4 GPU
# ============================================================

import torch
import subprocess

print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if not torch.cuda.is_available():
    raise RuntimeError(
        "CUDA is not available. Make sure the VS Code notebook "
        "is connected to the Colab T4 runtime."
    )

print("CUDA device:", torch.cuda.get_device_name(0))
print("CUDA capability:", torch.cuda.get_device_capability(0))

# Show NVIDIA GPU information
subprocess.run(["nvidia-smi"])

PyTorch version: 2.11.0+cu128
CUDA available: True
CUDA device: Tesla T4
CUDA capability: (7, 5)


CompletedProcess(args=['nvidia-smi'], returncode=0)

In [7]:
# ============================================================
# CELL 2 — CHECK COLAB /content FILESYSTEM
# ============================================================

from pathlib import Path

content = Path("/content")

print("Current /content contents:\n")

for item in sorted(content.iterdir()):
    print(item)

Current /content contents:

/content/.config
/content/dp_forgetbench_colab_utility
/content/dp_forgetbench_colab_utility.zip
/content/sample_data


In [15]:
# ============================================================
# CELL 3 — EXTRACT DP-FORGETBENCH BUNDLE
# ============================================================

from pathlib import Path
import zipfile

bundle_root = Path("/content/dp_forgetbench_colab_utility")
archives = sorted(Path("/content").glob("dp_forgetbench_colab_utility*.zip"))

if archives:
    archive_path = archives[-1]
    print("Extracting latest bundle:", archive_path)
    with zipfile.ZipFile(archive_path, "r") as archive:
        archive.extractall("/content")
    print("Extraction complete.")
elif bundle_root.exists():
    print("Bundle already extracted:", bundle_root)
else:
    raise FileNotFoundError(
        "Upload the latest dp_forgetbench_colab_utility.zip into the Colab /content Files sidebar."
    )

Extracting latest bundle: /content/dp_forgetbench_colab_utility.zip
Extraction complete.


In [16]:
# ============================================================
# CELL 4 — VERIFY PROJECT STRUCTURE
# ============================================================

from pathlib import Path

bundle_root = Path("/content/dp_forgetbench_colab_utility")

if not bundle_root.exists():
    raise FileNotFoundError(
        f"Bundle does not exist: {bundle_root}"
    )

print("Bundle:", bundle_root)
print("\nTop-level files/folders:\n")

for item in sorted(bundle_root.iterdir()):
    print("📁" if item.is_dir() else "📄", item.name)

Bundle: /content/dp_forgetbench_colab_utility

Top-level files/folders:

📁 colab
📁 configs
📁 data
📁 results
📁 scripts
📁 src
📁 tests


In [17]:
# 5. Install dependencies and run the current CNN non-private control
import os
from pathlib import Path

bundle_root = Path("/content/dp_forgetbench_colab_utility")
os.chdir(bundle_root)
!python -m pip install -q PyYAML scikit-learn dp-accounting
!python scripts/run_multiclass_utility_sweep.py --config configs/multiclass_cnn_nonprivate_control.yaml

Device: cuda (Tesla T4)
Model: small_groupnorm_cnn; epsilons=[inf]; rounds=[60]; seeds=[20260918, 20260919, 20260920]
epsilon=inf actual=inf seed=20260918 accuracy=46.00% [VALID]
epsilon=inf actual=inf seed=20260919 accuracy=44.63% [VALID]
epsilon=inf actual=inf seed=20260920 accuracy=43.83% [VALID]


In [18]:
# 6. Run the current CNN finite-epsilon diagnostic
import os
from pathlib import Path

bundle_root = Path("/content/dp_forgetbench_colab_utility")
os.chdir(bundle_root)
!python scripts/run_multiclass_utility_sweep.py --config configs/multiclass_cnn_dp_diagnostic.yaml

Device: cuda (Tesla T4)
Model: small_groupnorm_cnn; epsilons=[8.0, 16.0, 32.0]; rounds=[60]; seeds=[20260918, 20260919, 20260920]
epsilon=8 actual=8.0000 seed=20260918 accuracy=17.82% [INVALID]
epsilon=16 actual=16.0000 seed=20260918 accuracy=27.18% [WARNING]
epsilon=32 actual=32.0000 seed=20260918 accuracy=35.45% [WARNING]
epsilon=8 actual=8.0000 seed=20260919 accuracy=19.17% [INVALID]
epsilon=16 actual=16.0000 seed=20260919 accuracy=28.74% [WARNING]
epsilon=32 actual=32.0000 seed=20260919 accuracy=35.79% [WARNING]
epsilon=8 actual=8.0000 seed=20260920 accuracy=21.37% [INVALID]
epsilon=16 actual=16.0000 seed=20260920 accuracy=30.45% [WARNING]
epsilon=32 actual=32.0000 seed=20260920 accuracy=38.26% [WARNING]


In [ ]:
# 7. Test whether more rounds recover useful utility
import os
from pathlib import Path

bundle_root = Path("/content/dp_forgetbench_colab_utility")
os.chdir(bundle_root)
!python scripts/run_multiclass_utility_sweep.py --config configs/multiclass_cnn_rounds_ablation.yaml

Device: cuda (Tesla T4)
Model: small_groupnorm_cnn; epsilons=[8.0, 16.0, 32.0]; rounds=[40, 80, 120]; seeds=[20260918, 20260919, 20260920]
epsilon=8 actual=8.0000 seed=20260918 accuracy=20.51% [INVALID]
epsilon=16 actual=16.0000 seed=20260918 accuracy=31.45% [WARNING]
epsilon=32 actual=32.0000 seed=20260918 accuracy=37.97% [WARNING]
epsilon=8 actual=8.0000 seed=20260918 accuracy=18.48% [INVALID]
epsilon=16 actual=16.0000 seed=20260918 accuracy=26.63% [WARNING]
epsilon=32 actual=32.0000 seed=20260918 accuracy=35.88% [WARNING]
epsilon=8 actual=8.0000 seed=20260918 accuracy=12.60% [INVALID]
epsilon=16 actual=16.0000 seed=20260918 accuracy=19.78% [INVALID]
epsilon=32 actual=32.0000 seed=20260918 accuracy=32.50% [WARNING]
epsilon=8 actual=8.0000 seed=20260919 accuracy=22.09% [INVALID]
epsilon=16 actual=16.0000 seed=20260919 accuracy=29.28% [WARNING]
epsilon=32 actual=32.0000 seed=20260919 accuracy=33.90% [WARNING]


In [ ]:
# 8. Only after the CNN diagnosis: run the GroupNorm ResNet-18 sweep
import os
from pathlib import Path

bundle_root = Path("/content/dp_forgetbench_colab_utility")
os.chdir(bundle_root)
!python scripts/run_multiclass_utility_sweep.py --config configs/multiclass_resnet18_utility_sweep.yaml